In [1]:
# Load Data
import numpy as np
import pandas as pd

df = pd.read_csv("C:/Users/mayan/OneDrive/Desktop/Mayank(HP)/programs/Learning_AIML/Cleaned_data/DateFruit_Dataset.csv")

In [2]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [3]:
X = df.drop("Class",axis = 1)
y = df["Class"]

In [4]:
y.unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [7]:
from sklearn.preprocessing import StandardScaler , LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [9]:
from sklearn.model_selection import train_test_split 

X_train,X_test , y_train , y_test = train_test_split(
    X,y,test_size = 0.2, random_state = 42
)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Deep Learning

In [12]:
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import DataLoader ,TensorDataset

In [13]:
X_train_tensor = torch.tensor(X_train_scaled ,dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled ,dtype = torch.float32)

y_train_tensor = torch.tensor(y_train ,dtype = torch.long)
y_test_tensor = torch.tensor(y_test ,dtype = torch.long)

In [14]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [27]:
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle = True)
test_loader = DataLoader(test_dataset , batch_size = 32)

In [28]:
# Define Model 

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            nn.Linear(64 ,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )

    def forward(self , x):
        return self.model(x)

In [29]:
model = ANN()

# loss and Optimizers 
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [30]:
# Training the Neural Network 

epochs = 100 
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb,yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs , yb)
        loss.backward()
        optimizer.step() # Parameters updation
        
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    
    print(f"epoch = {epoch +1 } , loss = {train_loss}")

epoch = 1 , loss = 1.756237040395322
epoch = 2 , loss = 1.1608036652855251
epoch = 3 , loss = 0.750401689954426
epoch = 4 , loss = 0.5399775349575541
epoch = 5 , loss = 0.43739021990610205
epoch = 6 , loss = 0.37702198844888934
epoch = 7 , loss = 0.3381278501904529
epoch = 8 , loss = 0.3069206346636233
epoch = 9 , loss = 0.28202474829943286
epoch = 10 , loss = 0.2583120275774728
epoch = 11 , loss = 0.248702400404474
epoch = 12 , loss = 0.2400335054034772
epoch = 13 , loss = 0.2146619302423104
epoch = 14 , loss = 0.20808992081362268
epoch = 15 , loss = 0.19950833592725836
epoch = 16 , loss = 0.1905532375625942
epoch = 17 , loss = 0.1835017009921696
epoch = 18 , loss = 0.1821987805483134
epoch = 19 , loss = 0.18392016382321066
epoch = 20 , loss = 0.17236674674179242
epoch = 21 , loss = 0.16038825233345447
epoch = 22 , loss = 0.1601111422414365
epoch = 23 , loss = 0.15360020781340805
epoch = 24 , loss = 0.14693052459346212
epoch = 25 , loss = 0.14552864497122558
epoch = 26 , loss = 0.1469

In [32]:
# EValuate model 
model.eval()


total = 0
correct = 0

with torch.no_grad():
    for xb , yb in test_loader:
        outputs = model(xb)
        _ , predicted =  torch.max(outputs , 1) 

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch

print ("Accuracy  : ",correct/total * 100)

Accuracy  :  95.55555555555556
